# Лабораторная работа №6 — глубокие рекуррентные нейронные сети

Вариант 13: embedding `64`, два двунаправленных LSTM-слоя `32-32`; сравнение с Conv1D из трех сверточных слоев.

Код написан на PyTorch. Устройство выбирается автоматически: CUDA, если она доступна, иначе CPU.

## Импорты, настройки и константы

In [ ]:
from __future__ import annotations

import random
import re
import zipfile
from collections import Counter
from pathlib import Path


import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset, TensorDataset, random_split


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 13
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


TOKEN_RE = re.compile(r"[a-zA-Z']+")

## Функции и классы модели

In [ ]:
def tokenize(text: str) -> list[str]:
    return TOKEN_RE.findall(text.lower())

In [ ]:
def read_imdb(zip_path: Path, max_docs_per_class: int = 900) -> tuple[list[str], list[int]]:
    texts: list[str] = []
    labels: list[int] = []
    with zipfile.ZipFile(zip_path) as archive:
        for split in ["train", "test"]:
            for label_name, label in [("neg", 0), ("pos", 1)]:
                prefix = f"aclImdb/{split}/{label_name}/"
                names = [n for n in archive.namelist() if n.startswith(prefix) and n.endswith(".txt")]
                for name in names[:max_docs_per_class]:
                    texts.append(archive.read(name).decode("utf-8", errors="ignore"))
                    labels.append(label)
    return texts, labels

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts: list[str], labels: list[int], vocab: dict[str, int], max_len: int = 180):
        self.labels = labels
        self.samples = []
        for text in texts:
            ids = [vocab.get(token, 1) for token in tokenize(text)[:max_len]]
            self.samples.append(torch.tensor(ids or [1], dtype=torch.long))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.samples[index], torch.tensor(float(self.labels[index]), dtype=torch.float32)

In [ ]:
def collate_batch(batch: list[tuple[torch.Tensor, torch.Tensor]]) -> tuple[torch.Tensor, torch.Tensor]:
    texts, labels = zip(*batch)
    return pad_sequence(texts, batch_first=True, padding_value=0), torch.stack(labels)

In [ ]:
class BiLstmClassifier(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int = 64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.LSTM(
            input_size=embed_dim,
            hidden_size=32,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.2,
        )
        self.output = nn.Linear(64, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding(x)
        _, (hidden, _) = self.rnn(embedded)
        last_forward = hidden[-2]
        last_backward = hidden[-1]
        return self.output(torch.cat([last_forward, last_backward], dim=1)).squeeze(1)

In [ ]:
class ConvTextClassifier(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int = 64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.features = nn.Sequential(
            nn.Conv1d(embed_dim, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(1),
        )
        self.output = nn.Linear(64, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding(x).transpose(1, 2)
        features = self.features(embedded).squeeze(-1)
        return self.output(features).squeeze(1)

In [ ]:
def make_vocab(texts: list[str], max_words: int = 12000) -> dict[str, int]:
    counter = Counter()
    for text in texts:
        counter.update(tokenize(text))
    vocab = {"<pad>": 0, "<unk>": 1}
    for word, _ in counter.most_common(max_words - 2):
        vocab[word] = len(vocab)
    return vocab

In [ ]:
def train_binary(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, epochs: int = 4) -> dict[str, list[float]]:
    model.to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    history = {"loss": [], "val_loss": [], "acc": [], "val_acc": []}
    for _ in range(epochs):
        model.train()
        total_loss = correct = total = 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            total_loss += float(loss.item()) * len(x)
            correct += int(((torch.sigmoid(logits) >= 0.5).float() == y).sum().item())
            total += len(x)
        val_loss, val_acc = evaluate_binary(model, val_loader, criterion)
        history["loss"].append(total_loss / total)
        history["acc"].append(correct / total)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
    return history

In [ ]:
def evaluate_binary(model: nn.Module, loader: DataLoader, criterion: nn.Module) -> tuple[float, float]:
    model.eval()
    total_loss = correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            total_loss += float(criterion(logits, y).item()) * len(x)
            correct += int(((torch.sigmoid(logits) >= 0.5).float() == y).sum().item())
            total += len(x)
    return total_loss / total, correct / total

In [ ]:
def plot_history(history: dict[str, list[float]], path: Path) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(9, 3))
    axes[0].plot(history["loss"], label="train")
    axes[0].plot(history["val_loss"], label="val")
    axes[0].legend()
    axes[0].set_title("loss")
    axes[1].plot(history["acc"], label="train")
    axes[1].plot(history["val_acc"], label="val")
    axes[1].legend()
    axes[1].set_title("accuracy")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)

In [ ]:
def run_imdb() -> None:
    texts, labels = read_imdb(Path("IMDB/IMDB/aclImdb.zip"))
    vocab = make_vocab(texts)
    dataset = TextDataset(texts, labels, vocab)
    train_size = int(0.7 * len(dataset))
    val_size = int(0.15 * len(dataset))
    test_size = len(dataset) - train_size - val_size
    train_data, val_data, test_data = random_split(
        dataset, [train_size, val_size, test_size], generator=torch.Generator().manual_seed(SEED)
    )
    train_loader = DataLoader(train_data, batch_size=64, shuffle=True, collate_fn=collate_batch)
    val_loader = DataLoader(val_data, batch_size=128, collate_fn=collate_batch)
    test_loader = DataLoader(test_data, batch_size=128, collate_fn=collate_batch)

    lstm = BiLstmClassifier(len(vocab), embed_dim=64)
    history = train_binary(lstm, train_loader, val_loader, epochs=30)
    test_loss, test_acc = evaluate_binary(lstm, test_loader, nn.BCEWithLogitsLoss())
    plot_history(history, Path("lab_6/imdb_bilstm_history.png"))
    print(f"BiLSTM variant: test_loss={test_loss:.4f}, test_acc={test_acc:.4f}")

    conv = ConvTextClassifier(len(vocab), embed_dim=64)
    conv_history = train_binary(conv, train_loader, val_loader, epochs=30)
    conv_loss, conv_acc = evaluate_binary(conv, test_loader, nn.BCEWithLogitsLoss())
    plot_history(conv_history, Path("lab_6/imdb_conv_history.png"))
    print(f"Conv1D baseline: test_loss={conv_loss:.4f}, test_acc={conv_acc:.4f}")

In [ ]:
class ClimateLstm(nn.Module):
    def __init__(self, input_size: int):
        super().__init__()
        self.rnn = nn.LSTM(input_size, 64, num_layers=3, batch_first=True, bidirectional=True, dropout=0.3)
        self.output = nn.Linear(128, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        _, (hidden, _) = self.rnn(x)
        return self.output(torch.cat([hidden[-2], hidden[-1]], dim=1))

In [ ]:
def main() -> None:
    print(f"Device: {DEVICE}")
    run_imdb()
    climate_model = ClimateLstm(input_size=14)
    print(f"Jena Climate variant model parameters: {sum(p.numel() for p in climate_model.parameters())}")

## Запуск лабораторной

In [ ]:
main()